[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/IshtVibhu/Python-for-Computational-Economics/blob/main/NB_11_Econophysics.ipynb)

# Python for Computational Economics
## Notebook 11 — Econophysics: Power Laws, Fat Tails, and Complexity

**Prerequisites:** NB_01–NB_05

**What you will learn:**
- Power laws in wealth, city sizes, and firm sizes (Pareto/Zipf laws)
- Fitting power-law exponents with maximum likelihood
- Fat-tailed return distributions vs the Gaussian assumption
- GARCH(1,1) volatility clustering
- Entropy and information measures in economics
- The Sugarscape wealth distribution as an emergent power law

**What is econophysics?** Econophysics applies methods from statistical physics — scaling laws, phase transitions, universality — to economic systems. Key finding: many economic distributions are **power laws**, not Gaussian.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats, optimize

try:
    from arch import arch_model
except ImportError:
    !pip install arch --quiet
    from arch import arch_model

plt.style.use("seaborn-v0_8-whitegrid")
print("Libraries loaded.")

---
## 1. Power Laws — Pareto's Wealth Distribution

A power law distribution has PDF: `p(x) ∝ x^{-α}` for `x ≥ x_min`.

Pareto (1896) observed that wealth in Italy followed a power law with `α ≈ 2.5`. Modern studies find similar exponents worldwide.

In [ ]:
np.random.seed(42)

# Generate Pareto-distributed wealth (scipy uses the shape parameter b = alpha - 1)
true_alpha = 2.5
x_min      = 1.0
N          = 5000

wealth = stats.pareto.rvs(b=true_alpha - 1, scale=x_min, size=N)

# Maximum likelihood estimate of alpha (Hill estimator)
# For x > x_min: alpha_hat = n / sum(log(x_i / x_min))
x_min_threshold = np.percentile(wealth, 80)  # fit to top 20%
tail_data = wealth[wealth >= x_min_threshold]
alpha_hat = 1 / np.mean(np.log(tail_data / x_min_threshold)) + 1

print(f"True alpha:      {true_alpha:.2f}")
print(f"Hill estimator:  {alpha_hat:.2f}")
print(f"\nTop 1% wealth share:  {wealth[wealth >= np.percentile(wealth, 99)].sum() / wealth.sum():.1%}")
print(f"Top 10% wealth share: {wealth[wealth >= np.percentile(wealth, 90)].sum() / wealth.sum():.1%}")

# Log-log plot (power laws appear as straight lines in log-log space)
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].hist(wealth, bins=100, density=True, color="steelblue", alpha=0.7, range=(0, np.percentile(wealth, 98)))
axes[0].set_xlabel("Wealth")
axes[0].set_ylabel("Density")
axes[0].set_title("Wealth Distribution (linear scale)")

# Complementary CDF (CCDF) in log-log space
sorted_wealth = np.sort(wealth)
ccdf = 1 - np.arange(1, N+1) / N
axes[1].loglog(sorted_wealth, ccdf, ".", alpha=0.3, color="steelblue", markersize=2)
# Overlay theoretical power law
x_line = np.logspace(np.log10(x_min_threshold), np.log10(sorted_wealth.max()), 100)
ccdf_theory = (x_min_threshold / x_line) ** (true_alpha - 1)
ccdf_theory *= ccdf[np.searchsorted(sorted_wealth, x_min_threshold)]
axes[1].loglog(x_line, ccdf_theory, "r-", linewidth=2, label=f"α = {true_alpha:.1f}")
axes[1].set_xlabel("Wealth (log scale)")
axes[1].set_ylabel("P(W > w)")
axes[1].set_title("CCDF — Power law appears as straight line")
axes[1].legend()

plt.tight_layout()
plt.savefig("pareto_wealth.png", dpi=100, bbox_inches="tight")
plt.show()

---
## 2. Zipf's Law — City and Firm Sizes

Zipf (1949): the `k`-th largest city has population approximately proportional to `1/k`. This is a power law with exponent ≈ 1 in the rank-size relationship.

In [ ]:
# Simulated city populations following Zipf's law (Pareto with alpha ≈ 2)
np.random.seed(0)
N_CITIES = 500
city_pop = stats.pareto.rvs(b=1.0, scale=10_000, size=N_CITIES)
city_pop_sorted = np.sort(city_pop)[::-1]  # largest first

ranks = np.arange(1, N_CITIES + 1)

# Fit rank-size regression: log(size) = a - b*log(rank)
log_rank = np.log(ranks)
log_size = np.log(city_pop_sorted)
slope, intercept, r, p, se = stats.linregress(log_rank, log_size)

print(f"Rank-size regression: log(size) = {intercept:.2f} + ({slope:.3f}) * log(rank)")
print(f"Zipf exponent (|slope|): {-slope:.3f} (Zipf = 1.0 exactly)")
print(f"R² = {r**2:.3f}")

fig, ax = plt.subplots(figsize=(7, 5))
ax.loglog(ranks, city_pop_sorted, ".", alpha=0.5, color="steelblue", markersize=4, label="Cities")
ax.loglog(ranks, np.exp(intercept + slope * log_rank), "r-", linewidth=2,
           label=f"OLS fit (Zipf exponent = {-slope:.2f})")
ax.set_xlabel("Rank")
ax.set_ylabel("Population")
ax.set_title("Zipf's Law: City Rank-Size Distribution")
ax.legend()
plt.tight_layout()
plt.savefig("zipf_law.png", dpi=100, bbox_inches="tight")
plt.show()

---
## 3. Fat Tails in Financial Returns

The Gaussian (normal) distribution drastically underestimates the probability of large market moves. The **Student-t distribution** with a few degrees of freedom fits financial returns much better.

In [ ]:
np.random.seed(42)

# Simulate daily returns with fat tails (Student-t with 4 df)
df_true  = 4
n_obs    = 2000
scale    = 0.01  # daily volatility ~ 1%
returns  = stats.t.rvs(df=df_true, scale=scale, size=n_obs)

# Fit Gaussian and Student-t
mu_g, sigma_g = stats.norm.fit(returns)
df_t, loc_t, scale_t = stats.t.fit(returns)

print(f"Fitted Gaussian: μ = {mu_g:.5f}, σ = {sigma_g:.4f}")
print(f"Fitted Student-t: df = {df_t:.2f}, loc = {loc_t:.5f}, scale = {scale_t:.4f}")
print(f"True df = {df_true} — lower df = fatter tails")

# Kurtosis: Gaussian = 3 (excess kurtosis = 0); fat-tailed > 3
excess_kurtosis = stats.kurtosis(returns)
print(f"Excess kurtosis: {excess_kurtosis:.2f} (Gaussian = 0)")

# Plot the tails
x = np.linspace(-0.06, 0.06, 500)
fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(returns, bins=80, density=True, alpha=0.5, color="steelblue", label="Simulated returns")
ax.plot(x, stats.norm.pdf(x, mu_g, sigma_g), "r-", linewidth=2, label="Gaussian fit")
ax.plot(x, stats.t.pdf(x, df_t, loc_t, scale_t), "g-", linewidth=2, label=f"Student-t fit (df={df_t:.1f})")
ax.set_yscale("log")  # log scale to see the tails clearly
ax.set_xlabel("Daily Return")
ax.set_ylabel("Log Density")
ax.set_title("Fat Tails: Gaussian vs Student-t Fit")
ax.legend()
plt.tight_layout()
plt.savefig("fat_tails.png", dpi=100, bbox_inches="tight")
plt.show()

---
## 4. Volatility Clustering — GARCH(1,1)

Financial markets show **volatility clustering**: large moves cluster together. The GARCH(1,1) model captures this:

```
σ²_t = ω + α * ε²_{t-1} + β * σ²_{t-1}
```

where `ω + α + β < 1` for stationarity.

In [ ]:
np.random.seed(7)

# Simulate a GARCH(1,1) series
omega = 0.00001
alpha_g = 0.08
beta_g  = 0.90
n       = 2000

sigma2 = np.zeros(n)
eps    = np.zeros(n)
sigma2[0] = omega / (1 - alpha_g - beta_g)  # unconditional variance

for t in range(1, n):
    sigma2[t] = omega + alpha_g * eps[t-1]**2 + beta_g * sigma2[t-1]
    eps[t]    = np.sqrt(sigma2[t]) * np.random.normal()

# Fit GARCH(1,1)
garch = arch_model(eps * 100, vol="Garch", p=1, q=1, mean="Zero", dist="Normal")
garch_fit = garch.fit(disp="off")

print("GARCH(1,1) parameter estimates:")
print(garch_fit.params.round(6))

fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
axes[0].plot(eps * 100, linewidth=0.5, color="steelblue", alpha=0.8)
axes[0].set_ylabel("Return (%)")
axes[0].set_title("GARCH(1,1) — Simulated Returns")

axes[1].plot(np.sqrt(sigma2) * 100, linewidth=1, color="red", label="True vol")
axes[1].plot(garch_fit.conditional_volatility, linewidth=1, color="green", linestyle="--",
             label="GARCH estimated vol")
axes[1].set_ylabel("Volatility (%)")
axes[1].set_xlabel("Time")
axes[1].legend()

plt.tight_layout()
plt.savefig("garch_volatility.png", dpi=100, bbox_inches="tight")
plt.show()

---
## 5. Emergent Power Law in Sugarscape

The Sugarscape model, despite using only simple local rules, generates a **right-skewed wealth distribution** that resembles a power law — replicating Pareto's empirical finding from simulation.

In [ ]:
np.random.seed(42)

# Simple multiplicative wealth process (stylised Sugarscape)
N_AGENTS = 1000
N_STEPS  = 100
wealth   = np.full(N_AGENTS, 20.0)

for step in range(N_STEPS):
    # Harvest: drawn from right-skewed landscape
    harvest = np.random.exponential(scale=2.0, size=N_AGENTS)
    # Trade: multiplicative random gain/loss
    trade_factor = np.random.lognormal(0, 0.25, N_AGENTS)
    # Metabolism: drawn from discrete uniform
    metabolism = np.random.randint(1, 4, N_AGENTS)

    wealth = wealth * trade_factor + harvest - metabolism
    wealth = np.maximum(wealth, 0.1)  # no bankruptcy below minimum

# Hill estimator for power-law exponent
x_min_q = np.percentile(wealth, 80)
tail = wealth[wealth > x_min_q]
alpha_hill = 1 + len(tail) / np.sum(np.log(tail / x_min_q))

print(f"Sugarscape wealth after {N_STEPS} steps:")
print(f"  Mean:   {wealth.mean():.1f}")
print(f"  Median: {np.median(wealth):.1f}")
print(f"  Max:    {wealth.max():.1f}")
print(f"  Hill estimator (tail alpha): {alpha_hill:.2f}")
print(f"  Comparison: empirical wealth distributions typically have alpha in 1.5–3.0")

# CCDF log-log plot
sorted_w = np.sort(wealth)
ccdf = 1 - np.arange(1, N_AGENTS + 1) / N_AGENTS

fig, ax = plt.subplots(figsize=(7, 5))
ax.loglog(sorted_w, ccdf, ".", alpha=0.4, color="steelblue", markersize=3, label="Sugarscape")
# Reference line
x_ref = np.linspace(x_min_q, sorted_w.max(), 100)
y_ref = (x_min_q / x_ref) ** (alpha_hill - 1) * ccdf[np.searchsorted(sorted_w, x_min_q)]
ax.loglog(x_ref, y_ref, "r--", linewidth=2, label=f"Power law α = {alpha_hill:.2f}")
ax.set_xlabel("Wealth")
ax.set_ylabel("P(W > w)")
ax.set_title("Emergent Power Law in Sugarscape Wealth Distribution")
ax.legend()
plt.tight_layout()
plt.savefig("sugarscape_powerlaw.png", dpi=100, bbox_inches="tight")
plt.show()

---
## Your Turn — Does More Trade Increase the Power-Law Exponent?

Re-run the Sugarscape simulation above, but vary the `sigma` parameter of the log-normal trade factor: try `sigma ∈ {0.05, 0.15, 0.25, 0.40}`. For each, compute the Hill estimator. Does more volatile trade (higher sigma) lead to a smaller exponent (fatter tail = more inequality)?

In [ ]:
# Your code here


In [ ]:
#@title Solution
print("Effect of trade volatility on wealth inequality:")
print(f"{'Sigma':<10} {'Hill alpha':<12} {'Top 10% share':<15}")
print("-" * 37)

for sigma_trade in [0.05, 0.15, 0.25, 0.40]:
    np.random.seed(42)
    wealth_s = np.full(N_AGENTS, 20.0)
    for _ in range(N_STEPS):
        harvest = np.random.exponential(2.0, N_AGENTS)
        trade_factor = np.random.lognormal(0, sigma_trade, N_AGENTS)
        metabolism = np.random.randint(1, 4, N_AGENTS)
        wealth_s = np.maximum(wealth_s * trade_factor + harvest - metabolism, 0.1)
    x_min_s = np.percentile(wealth_s, 80)
    tail_s   = wealth_s[wealth_s > x_min_s]
    alpha_s  = 1 + len(tail_s) / np.sum(np.log(tail_s / x_min_s))
    top10 = wealth_s[wealth_s >= np.percentile(wealth_s, 90)].sum() / wealth_s.sum()
    print(f"{sigma_trade:<10.2f} {alpha_s:<12.3f} {top10:<15.1%}")

print("\nHigher trade volatility → smaller alpha → fatter tail → more inequality.")

---
## Summary

| Concept | How to measure / fit |
|---|---|
| Power law exponent | Hill estimator: `1 + n / Σlog(x_i/x_min)` |
| CCDF log-log plot | `plt.loglog(sorted_x, 1 - ecdf)` |
| Fat-tail fit | `scipy.stats.t.fit(returns)` |
| Excess kurtosis | `scipy.stats.kurtosis(data)` |
| GARCH(1,1) | `arch_model(returns, vol='Garch').fit()` |
| Zipf rank-size | `scipy.stats.linregress(log_rank, log_size)` |

**References:**
- Mantegna, R. N., & Stanley, H. E. (1999). *An Introduction to Econophysics.* Cambridge.
- Gabaix, X. (2009). Power laws in economics and finance. *Annual Review of Economics*, 1, 255–294.
- Pareto, V. (1896). *Cours d'économie politique.*

---
## Series Complete

You have now completed all 12 foundational notebooks:

| Notebook | Topic |
|---|---|
| NB_00 | Overview and guide |
| NB_01 | Python fundamentals |
| NB_02 | NumPy |
| NB_03 | Pandas |
| NB_04 | Matplotlib & Seaborn |
| NB_05 | SciPy |
| NB_06 | StatsModels & linearmodels |
| NB_07 | QuantEcon |
| NB_08 | Mesa ABM |
| NB_09 | NetworkX |
| NB_10 | Financial Data |
| NB_11 | Econophysics (this notebook) |

**Next step:** Explore the full Sugarscape implementation in the companion **Session_3 through Session_20** notebooks, where every concept from this series comes together.